# Pipeline test bench

Runs the whole pipeline (Agents 0-8) on **one paper**, then lets you swap the **embedding model** and the **inference model** to see what changes.

**How to use:** edit the settings cell, then *Run All*. Nothing prompts you; a bad setting stops the run and says how to fix it.

| Stage | Agent | Depends on |
|---|---|---|
| Find the seed paper, extract and fetch its references | 0, 1, 2 | nothing you can change (done once) |
| Index the papers | 3 (6 if you add a PDF) | the **embedding model**: one corpus each |
| Retrieval comparison | | the embedding model only, no LLM |
| Synthesis, cite a sentence, cite a draft, verify, chat | 4, 5, 8, 7 | embedding model x **inference model** |

**Held fixed so the comparison is fair:** the same downloaded PDFs for every corpus; index v2 (sentence-window chunks, so chunk boundaries do not depend on the embedder); one summary model for every corpus (the first in `LLMS`).

**Needs:** the environment from `requirements.txt` plus `jupyter`; [Ollama](https://ollama.com) running with at least one embedding model and one chat model (`ollama pull <name>`; this notebook never downloads a model). To read the seed's reference list it also needs **GROBID** (`docker run --rm -d --name grobid -p 8070:8070 grobid/grobid:0.8.1`) or, as a weaker fallback, poppler's `pdftotext`. Only GROBID gives DOIs, authors and years, and only GROBID enables the retrieval scores against the paper's own citations.

**Safe by construction:** everything is written under `notebooks/.data/`, and the notebook refuses to use the project's own `data/` directory.

**Time:** dominated by local model calls. The default seed has 6 references, so expect tens of minutes; a seed with 60 references takes hours.

In [ ]:
import os
from pathlib import Path

# Conda environments that carry both torch and faiss/sklearn abort on import with an OpenMP
# "already initialized" error. config.py imports torch, so this must be set before anything else.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# ── Settings: edit, then Run All ─────────────────────────────────────────────
SEED = "arXiv:0909.0418"   # an arXiv id or link, or the path to a PDF. Run time grows with the seed's reference list.
QUESTION = "What is known about log-periodic patterns that precede stock market crashes?"   # for the synthesis and the chat turn
DRAFT = (                  # sentences to cite (Agent 5) and verify (Agent 8)
    "Financial markets often display log-periodic oscillations before a major crash. "
    "The preferred scaling factor of these oscillations is close to two. "
    "A bubble can end in a sharp correction when the log-periodic pattern reaches its critical time. "
    "Forecasting models based on log-periodic self-similarity have been applied to the S&P500 and other world indices."
)
EMBEDDERS = None           # None = every installed embedding model (at most 3), or a list: ["nomic-embed-text", "mxbai-embed-large"]
LLMS = None                # None = the first installed chat model, or a list: ["gemma4:e2b", "qwen2.5:7b"]; the first also writes every corpus's paper summaries
MANUAL_PDF = ""            # optional: a PDF to add to every corpus by hand (Agent 6)
LLM_BACKEND = "ollama"     # or "openai" (any OpenAI-compatible endpoint: set OPENAI_BASE_URL / OPENAI_API_KEY before starting Jupyter, and list EMBEDDERS and LLMS)
EMBED_BACKEND = "ollama"

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

In [ ]:
import contextlib
import os
import re
import sys
import time
from pathlib import Path

timings = {}   # stage name -> seconds, filled in by stage()


def check_data_dir(path):
    """The pipeline writes indexes, downloads and logs. Never let that be the project's own data/."""
    real, path = (REPO / "data").resolve(), Path(path).resolve()
    if path == real or real in path.parents:
        raise RuntimeError(f"{path} is inside the project's own data/ directory. Use a scratch folder.")


def slug(name):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", name)


def use(embed, llm, data_dir):
    """Point the pipeline at one (embedding model, inference model, corpus) combination.

    config.py reads models and paths when it is imported and other modules copy them by name,
    so: set the environment, forget every project module, import afresh."""
    check_data_dir(data_dir)
    for name in [m for m in sys.modules if m in ("orchestrate", "research_assistant") or m.startswith("research_assistant.")]:
        del sys.modules[name]
    for key in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):   # a stray key would send chat and the judge to Gemini
        os.environ.pop(key, None)
    os.environ.update({
        "CITATION_DATA_DIR": str(data_dir), "CITATION_INDEX_VERSION": "2", "CITATION_LOG_FILE": "0",
        "LLM_BACKEND": LLM_BACKEND, "CITATION_EMBED_BACKEND": EMBED_BACKEND,
        "CITATION_EMBED_MODEL": embed, "CITATION_LLM_MODEL": llm, "CITATION_CHAT_MODEL": llm,
        "CITATION_JUDGEMENT_MODEL": llm, "CITATION_SUMMARY_MODEL": SUMMARY_MODEL,
    })


@contextlib.contextmanager
def stage(name):
    start = time.perf_counter()
    print(f"> {name}")
    yield
    timings[name] = time.perf_counter() - start
    print(f"ok {name}  ({timings[name]:.0f}s)")

In [ ]:
import json
import shutil
import urllib.error
import urllib.request

sys.path.insert(0, str(REPO))                  # so `import orchestrate` finds the project
DATA = REPO / "notebooks" / ".data"
SHARED = DATA / "shared"                       # Agents 0-2: nothing in here depends on a model
check_data_dir(DATA)

if not QUESTION.strip():
    raise ValueError("Set QUESTION: the synthesis and the chat turn need a research question.")


def ollama(path, body=None, timeout=10):
    host = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
    request = urllib.request.Request(
        (host if host.startswith("http") else "http://" + host) + path,
        json.dumps(body).encode() if body else None, {"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:      # Ollama explains itself, e.g. "model requires 6.9 GiB but only 4.8 GiB are available"
        raise RuntimeError(f"Ollama refused {path}: {error.read().decode()[:300]}") from error


if "ollama" in (LLM_BACKEND, EMBED_BACKEND):
    try:
        installed = {m["name"]: ollama("/api/show", {"model": m["name"]}).get("capabilities") or []
                     for m in ollama("/api/tags")["models"]}
    except urllib.error.URLError as error:
        raise RuntimeError(f"Ollama is not reachable ({error}). Start it with `ollama serve`.") from error
    if EMBED_BACKEND == "ollama":
        EMBEDDERS = EMBEDDERS or [n for n, caps in installed.items() if "embedding" in caps][:3]
    if LLM_BACKEND == "ollama":
        LLMS = LLMS or [n for n, caps in installed.items() if "completion" in caps][:1]
if not (EMBEDDERS and LLMS):
    raise ValueError("Nothing to compare. Pull models into Ollama (`ollama pull nomic-embed-text`, `ollama pull <chat model>`) or list EMBEDDERS and LLMS.")
SUMMARY_MODEL = LLMS[0]

# Each Ollama model must exist, be the right kind, and actually load. A model that does not fit in memory fails here, not ten cells later.
for model, kind, backend in [(m, "embedding", EMBED_BACKEND) for m in EMBEDDERS] + [(m, "completion", LLM_BACKEND) for m in LLMS]:
    if backend != "ollama":
        continue
    match = next((n for n in installed if n in (model, model + ":latest")), None)
    if match is None:
        raise RuntimeError(f"{model!r} is not installed. Run: ollama pull {model}")
    if kind not in installed[match]:
        raise RuntimeError(f"{model!r} is not a {kind} model (it offers {installed[match]})")
    if kind == "embedding":
        ollama("/api/embed", {"model": model, "input": "ping"}, timeout=300)
    else:
        ollama("/api/chat", {"model": model, "stream": False, "messages": [{"role": "user", "content": "ping"}],
                             "options": {"num_predict": 1}}, timeout=300)
    print(f"ok  {model} ({kind}) loads and answers")

grobid = os.environ.get("GROBID_SERVER", "http://localhost:8070")
try:
    urllib.request.urlopen(grobid + "/api/isalive", timeout=3).close()
    grobid_note = "up"
except OSError:
    grobid_note = "down: references are read by regex via pdftotext (no DOIs, fewer downloads) and the retrieval scores against the paper's own citations are skipped"
    if shutil.which("pdftotext") is None:
        raise RuntimeError(
            "Agent 1 needs GROBID or poppler's pdftotext to read the reference list, and neither is available.\n"
            "  GROBID (better: DOIs, authors, years, and the retrieval scores): docker run --rm -d --name grobid -p 8070:8070 grobid/grobid:0.8.1\n"
            "  poppler (regex fallback): brew install poppler   or   apt install poppler-utils")

print(f"seed       {SEED}")
print(f"question   {QUESTION}")
print(f"embedders  {EMBEDDERS}")
print(f"inference  {LLMS}   (the first also writes the summaries)")
print(f"backends   {LLM_BACKEND} / {EMBED_BACKEND}")
print(f"data       {DATA}")
print(f"GROBID     {grobid_note}")

## Shared stages (Agents 0-2)

None of these use a model you can change, so they run once and every corpus below is built from the same papers.

In [ ]:
use(EMBEDDERS[0], LLMS[0], SHARED)
import orchestrate as orch

seed = SEED.strip()
state = {"query": QUESTION, "workers": 1, "force": False, "ask": True, "seed_url": None, "seed_file": None,
         "describe_figures": False, "audit_citations": False}
if Path(seed).expanduser().is_file():
    state["seed_file"] = str(Path(seed).expanduser().resolve())
elif seed.lower().startswith("arxiv:"):
    state["seed_url"] = "https://arxiv.org/abs/" + seed[len("arxiv:"):].strip()
elif seed.startswith("http"):
    state["seed_url"] = seed
else:
    raise ValueError("Set SEED to an arXiv id such as 'arXiv:0909.0418', a link, or the path to a PDF.")

with stage("Agent 0 | find the seed paper"):
    state.update(orch.discover(state))
    assert state.get("seed_path") and os.path.exists(state["seed_path"]), state.get("stopped") or "no seed PDF"
    print("seed paper:", state["seed_label"])

In [ ]:
with stage("Agent 1 | extract the seed's reference list"):
    state.update(orch.extract(state))
    assert state.get("references_ok"), state.get("stopped")
    seed = Path(state["seed_path"])
    for folder in ("processed", "failed"):                   # Agent 1 files the PDF away once it has read it
        if (seed.parent / folder / seed.name).exists():
            state["seed_path"] = str(seed.parent / folder / seed.name)
    assert os.path.exists(state["seed_path"]), "the seed PDF went missing"
    print({k: v for k, v in state["extraction"].items() if not isinstance(v, (list, dict))})

In [ ]:
with stage("Agent 2 | fetch the open-access references"):
    orch.fetch(state)
    from research_assistant.config import DOWNLOADED_JSON_PATH
    downloaded = json.loads(Path(DOWNLOADED_JSON_PATH).read_text())
    assert downloaded, "no reference could be downloaded: paywalled, or the network is blocked"
    print(f"{len(downloaded)} reference PDFs downloaded; the seed lists {state['extraction'].get('reference_count')} references")

## One corpus per embedding model (Agent 3)

The embedding model has to stay fixed for a corpus, so each embedder gets its own. They share the downloaded PDFs (symlinked) and start from copies of the same manifests.

In [ ]:
corpora, built = {}, {}
for E in EMBEDDERS:
    corpus = corpora[E] = DATA / slug(E)
    corpus.mkdir(parents=True, exist_ok=True)
    for name in ("raw", "pulled_pdfs"):                      # shared, so every corpus holds the same papers
        if not (corpus / name).exists():
            (corpus / name).symlink_to(SHARED / name)
    for name in ("downloaded.json", "extracted_citations.json", "seed_papers.json"):
        if (SHARED / name).exists():
            shutil.copy(SHARED / name, corpus / name)        # absolute paths inside, so they stay valid
    use(E, LLMS[0], corpus)
    import orchestrate as orch
    from research_assistant.shared.db import load_search_resources
    from research_assistant.shared.retrieve import document_summaries
    with stage(f"Agent 3 | {E} | index the papers"):
        orch.ingest_seed(state)
        orch.ingest_refs(state)
        _, _, texts, metadatas = load_search_resources()
        documents = {m["document"] for m in metadatas}
        built[E] = {"chunks": len(texts), "documents": len(documents), "summaries": len(document_summaries(documents))}
        assert os.path.basename(state["seed_path"]) in documents, f"the seed paper was not indexed: {sorted(documents)}"
        assert built[E]["chunks"] > 0 and built[E]["documents"] >= 2, built[E]    # the seed plus at least one reference
        assert built[E]["summaries"] == built[E]["documents"], f"{built[E]}: the summary model failed on some papers (see the log above)"
        print(E, built[E])

In [ ]:
# Agent 6: a paper you downloaded by hand, e.g. a paywalled reference. Optional.
if MANUAL_PDF:
    for E in EMBEDDERS:
        use(E, LLMS[0], corpora[E])
        from research_assistant.agents.agent6_manual_ingestor import ingest_manual_pdf
        with stage(f"Agent 6 | {E} | add {Path(MANUAL_PDF).name} by hand"):
            result = ingest_manual_pdf(str(Path(MANUAL_PDF).expanduser()), citation_string=Path(MANUAL_PDF).stem)
            assert result is not None, "the PDF was not ingested"
else:
    print("Agent 6 skipped: set MANUAL_PDF to add a PDF to every corpus by hand")

## Retrieval, per embedding model (no LLM involved)

If GROBID produced a TEI for the seed, each sentence the authors cited becomes a query and the score is whether the cited paper is in the top 5 (`hit@5`) and how high (`MRR`). Author citations are a floor, not truth: a correct paper the authors did not cite counts as a miss. Without a TEI the probe queries are your `DRAFT` sentences, and the comparison is how much the top-5 papers overlap between embedders.

In [ ]:
retrieval = {}
use(EMBEDDERS[0], LLMS[0], corpora[EMBEDDERS[0]])
from research_assistant.eval import citer_gold
try:
    gold = [c for c in citer_gold.build(state["seed_path"]) if c["kind"] == "cited"]
except FileNotFoundError:
    gold = []                                                # no GROBID TEI for the seed
sentences = re.split(r"(?<=[.!?])\s+", DRAFT.strip())
queries = [c["sentence"] for c in gold] or sentences

for E in EMBEDDERS:
    use(E, LLMS[0], corpora[E])
    from research_assistant.shared.db import load_search_resources
    from research_assistant.shared.search import hybrid_search
    resources = load_search_resources()
    with stage(f"Retrieval | {E} | no LLM"):
        top_docs = []
        for query in queries:
            hits = hybrid_search(query, *resources, top_k=5, exclude_docs={os.path.basename(state["seed_path"])})
            ordered = []
            for hit in hits:
                if hit["metadata"]["document"] not in ordered:
                    ordered.append(hit["metadata"]["document"])
            top_docs.append(ordered)
        retrieval[E] = {"top_docs": top_docs}
        if gold:
            ranks = [next((i for i, d in enumerate(docs, 1) if d in case["author_documents"]), None)
                     for case, docs in zip(gold, top_docs)]
            retrieval[E]["hit_at_5"] = sum(r is not None for r in ranks) / len(ranks)
            retrieval[E]["mrr"] = sum(1 / r for r in ranks if r) / len(ranks)
        assert len(top_docs) == len(queries) and all(top_docs), "retrieval returned nothing for some query"
        print(E, {k: round(v, 2) for k, v in retrieval[E].items() if k != "top_docs"} or f"{len(queries)} probe queries")

## Embedding model x inference model

Each pair runs the same stages on the same papers: the related-work synthesis, one sentence cited by Agent 4, the whole draft cited by Agent 5, every citation judged by Agent 8, and one chat turn by Agent 7.

In [ ]:
runs = {}
for E in EMBEDDERS:
    for L in LLMS:
        use(E, L, corpora[E])
        from research_assistant.shared.retrieve import research_answer
        with stage(f"Synthesis | {E} | {L}"):
            answer = research_answer(QUESTION)
            assert answer and answer["suggestion"].strip(), "empty synthesis"
            runs[(E, L)] = {"synthesis": answer}
            print(f"{len(answer['suggestion'])} characters from {len(answer['keys'])} papers; unverified keys: {answer['unverified_citations']}")

In [ ]:
for E in EMBEDDERS:
    for L in LLMS:
        use(E, L, corpora[E])
        from research_assistant.agents.agent4_assistant import suggest_citation
        with stage(f"Agent 4 | {E} | {L} | cite one sentence"):
            suggestion = suggest_citation(sentences[0])
            assert suggestion and suggestion["suggestion"].strip(), "no suggestion"
            runs[(E, L)]["agent4"] = suggestion["suggestion"]
            print(suggestion["suggestion"][:300])

In [ ]:
for E in EMBEDDERS:
    for L in LLMS:
        use(E, L, corpora[E])
        from research_assistant.agents.agent5_batch_citer import run_batch_citer
        work = corpora[E] / "runs" / slug(L)
        work.mkdir(parents=True, exist_ok=True)
        (work / "draft.txt").write_text(DRAFT)
        with stage(f"Agent 5 | {E} | {L} | cite the draft"):
            out = run_batch_citer(str(work / "draft.txt"), str(work / "cited.txt"))
            assert out and Path(out).exists(), "the citer wrote no draft"
            cited = Path(out).read_text()
            runs[(E, L)].update(work=work, cited=cited, citations=cited.count("\\cite{"))
            print(f"{runs[(E, L)]['citations']} citations inserted")

In [ ]:
for E in EMBEDDERS:
    for L in LLMS:
        use(E, L, corpora[E])
        from research_assistant.agents.agent8_verifier import verify_draft
        with stage(f"Agent 8 | {E} | {L} | verify the citations"):
            report = verify_draft(str(runs[(E, L)]["work"] / "cited.txt"))
            assert report and report["totals"], "no verification record"
            runs[(E, L)]["verification"] = report
            print(report["totals"])

In [ ]:
for E in EMBEDDERS:
    for L in LLMS:
        use(E, L, corpora[E])
        from research_assistant.agents.agent7_research_chat import ResearchChat
        with stage(f"Agent 7 | {E} | {L} | chat"):
            agent = ResearchChat(top_k=5)
            reply = "".join(agent.stream_turn(QUESTION))
            assert reply.strip() and agent.last_sources, "no grounded answer"
            runs[(E, L)].update(chat=reply, chat_warnings=agent.last_warnings)
            print(reply[:300])

## Comparison

Timings are wall-clock on this machine and include model load time on the first call. Verdict counts come from Agent 8's own totals. Everything here is also saved to `notebooks/.data/results.json`.

In [ ]:
def seconds(prefix, *parts):
    return timings.get(f"{prefix} | " + " | ".join(parts), float("nan"))


print("EMBEDDING MODELS (one corpus each)")
print(f"{'model':30} {'index s':>8} {'papers':>7} {'chunks':>7} {'hit@5':>6} {'MRR':>6} {'top-5 overlap with first':>26}")
first = retrieval[EMBEDDERS[0]]["top_docs"]
for E in EMBEDDERS:
    r = retrieval[E]
    overlap = sum(len(set(a) & set(b)) / len(set(a) | set(b)) for a, b in zip(r["top_docs"], first)) / len(first)
    hit, mrr = r.get("hit_at_5"), r.get("mrr")
    print(f"{E:30} {seconds('Agent 3', E, 'index the papers'):8.0f} {built[E]['documents']:7} {built[E]['chunks']:7} "
          f"{'n/a' if hit is None else format(hit, '.2f'):>6} {'n/a' if mrr is None else format(mrr, '.2f'):>6} {overlap:26.2f}")

print()
print("EMBEDDING x INFERENCE MODEL")
print(f"{'embedder':24} {'model':22} {'synth s':>7} {'chars':>6} {'cites':>5} {'cite s':>6} {'verify s':>8} {'chat s':>6} {'chat chars':>10}")
for (E, L), run in runs.items():
    print(f"{E:24} {L:22} {seconds('Synthesis', E, L):7.0f} {len(run['synthesis']['suggestion']):6} {run['citations']:5} "
          f"{seconds('Agent 5', E, L, 'cite the draft'):6.0f} {seconds('Agent 8', E, L, 'verify the citations'):8.0f} "
          f"{seconds('Agent 7', E, L, 'chat'):6.0f} {len(run['chat']):10}")
    print(f"{'':24} verdicts: {run['verification']['totals']}")

for (E, L), run in runs.items():
    print(f"\n{'=' * 78}\n{E}  x  {L}\n{'=' * 78}")
    print("SYNTHESIS\n" + run["synthesis"]["suggestion"])
    print("\nONE SENTENCE (Agent 4)\n" + run["agent4"])
    print("\nCITED DRAFT (Agent 5)\n" + run["cited"])
    print("\nCHAT (Agent 7)\n" + run["chat"])

(DATA / "results.json").write_text(json.dumps(
    {"seed": SEED, "question": QUESTION, "timings": timings, "built": built,
     "retrieval": {E: {k: v for k, v in r.items() if k != "top_docs"} for E, r in retrieval.items()},
     "runs": {f"{E} x {L}": {k: v for k, v in run.items() if k != "work"} for (E, L), run in runs.items()}},
    default=str, indent=1))
print(f"\nsaved {DATA / 'results.json'}")